# 🧬 DNABERT-2 Genomic Variant Classifier
**End-to-end pipeline**: ClinVar CSV → Sequence Extraction → Fine-tune DNABERT-2 → Inference

**Runtime**: ~25 min on T4 GPU

---

## 1. Install Dependencies

In [ ]:
!pip install -q transformers==4.40.0 datasets accelerate pyfaidx scikit-learn einops
!pip install -q triton 2>/dev/null || echo 'triton not available, continuing without flash attention'

## 2. Upload Dataset

In [ ]:
import os, random, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    classification_report, confusion_matrix,
)
from transformers import (
    AutoTokenizer, AutoModel,
    TrainingArguments, Trainer,
)
from transformers.modeling_outputs import SequenceClassifierOutput
from torch.utils.data import Dataset

warnings.filterwarnings('ignore')

# ── Reproducibility ──
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# ── Config ──
MODEL_NAME   = 'zhihan1996/DNABERT-2-117M'
CONTEXT      = 64          # bases each side → 129bp total sequence
MAX_LENGTH   = 128         # tokenizer max length
BATCH_SIZE   = 16
EPOCHS       = 3
LR           = 2e-5

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}', end='')
if device == 'cuda':
    print(f' — {torch.cuda.get_device_name(0)}')
else:
    print(' ⚠️ No GPU detected — training will be very slow')

In [ ]:
from google.colab import files
print('Upload clinvar_balanced_snps.csv:')
uploaded = files.upload()
CSV_FILE = list(uploaded.keys())[0]

df = pd.read_csv(CSV_FILE)
print(f'\n✓ Loaded {len(df):,} rows')
print(f'Columns: {list(df.columns)}')
print(f'\nClass distribution:')
print(df['label'].value_counts().to_string())
df.head()

## 3. Download GRCh38 Reference Genome
Downloads the human reference genome (~938 MB compressed). Takes ~5–8 min.

In [ ]:
GENOME_URL = 'https://hgdownload.soe.ucsc.edu/goldenPath/hg38/bigZips/hg38.fa.gz'
GENOME_FA  = 'hg38.fa'

if not os.path.exists(GENOME_FA):
    print('⬇️  Downloading GRCh38 (~938 MB)...')
    !wget -q --show-progress $GENOME_URL -O hg38.fa.gz
    print('📦 Decompressing (~2 min)...')
    !gunzip -f hg38.fa.gz
    print('✓ Genome ready')
else:
    print('✓ Genome already present')

from pyfaidx import Fasta
print('📖 Indexing genome (first time takes ~2 min)...')
genome = Fasta(GENOME_FA, rebuild=False)
print(f'✓ Indexed — {len(genome.keys())} contigs loaded')

## 4. Sequence Extraction
For each variant, extract ±64 bp around the position and substitute the alternate allele.

In [ ]:
VALID_BASES = set('ACGT')

# Build chromosome name lookup (CSV uses '1', UCSC uses 'chr1')
genome_keys = set(genome.keys())

def resolve_chrom(chrom_raw):
    """Map CSV chromosome name to FASTA contig name."""
    c = str(chrom_raw)
    # Try exact match first
    if c in genome_keys:
        return c
    # UCSC prefix: '1' → 'chr1'
    prefixed = f'chr{c}'
    if prefixed in genome_keys:
        return prefixed
    # Mitochondrial: ClinVar uses 'MT', UCSC uses 'chrM'
    if c == 'MT' and 'chrM' in genome_keys:
        return 'chrM'
    return None

def extract_sequence(row):
    """Extract mutant DNA sequence context around a variant."""
    chrom_key = resolve_chrom(row['Chromosome'])
    if chrom_key is None:
        return None

    pos = int(row['Start']) - 1                 # 0-based
    alt = str(row['AlternateAllele']).upper()
    if alt not in VALID_BASES:
        return None

    chrom_seq = genome[chrom_key]
    start = pos - CONTEXT
    end   = pos + CONTEXT + 1

    if start < 0 or end > len(chrom_seq):
        return None

    left  = str(chrom_seq[start:pos]).upper()
    right = str(chrom_seq[pos + 1:end]).upper()
    seq   = left + alt + right

    if not all(b in VALID_BASES for b in seq):
        return None
    return seq

print('🧬 Extracting sequences (±64 bp context)...')
df['sequence'] = df.apply(extract_sequence, axis=1)

before = len(df)
df = df.dropna(subset=['sequence']).reset_index(drop=True)
print(f'✓ {len(df):,} sequences extracted  ({before - len(df)} dropped)')
print(f'  Sequence length: {len(df["sequence"].iloc[0])} bp')
print(f'  Example: {df["sequence"].iloc[0][:40]}...')

## 5. Data Preparation

In [ ]:
# Re-balance after any dropped rows
min_class = df['label'].value_counts().min()
df = (
    df.groupby('label', group_keys=False)
      .apply(lambda x: x.sample(min_class, random_state=SEED))
      .sample(frac=1, random_state=SEED)
      .reset_index(drop=True)
)
print(f'Balanced dataset: {len(df):,} total')
print(df['label'].value_counts().to_string())

# Train / Test split
train_df, test_df = train_test_split(
    df, test_size=0.2, stratify=df['label'], random_state=SEED
)
print(f'\nTrain: {len(train_df):,}  |  Test: {len(test_df):,}')

## 6. Tokenization

In [ ]:
print('Loading DNABERT-2 tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
print(f'✓ Vocab size: {tokenizer.vocab_size:,}')

class DNADataset(Dataset):
    """Tokenized DNA sequence dataset for Trainer."""
    def __init__(self, sequences, labels):
        self.encodings = tokenizer(
            sequences,
            max_length=MAX_LENGTH,
            padding='max_length',
            truncation=True,
            return_tensors='pt',
        )
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        item['labels'] = self.labels[idx]
        return item

train_dataset = DNADataset(train_df['sequence'].tolist(), train_df['label'].tolist())
test_dataset  = DNADataset(test_df['sequence'].tolist(),  test_df['label'].tolist())
print(f'✓ Train: {len(train_dataset):,} samples  |  Test: {len(test_dataset):,} samples')

## 7. Model
Wraps the DNABERT-2 encoder with a classification head (dropout + linear).

In [ ]:
class DNABertClassifier(nn.Module):
    """DNABERT-2 encoder + classification head."""

    def __init__(self, model_name, num_labels=2, dropout=0.1):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(
            model_name, trust_remote_code=True
        )
        hidden = self.encoder.config.hidden_size
        self.head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden, num_labels),
        )
        # Trainer checks model.config
        self.config = self.encoder.config

    def forward(self, input_ids, attention_mask=None, labels=None, **kw):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls_emb = out.last_hidden_state[:, 0]       # [CLS] token
        logits  = self.head(cls_emb)

        loss = None
        if labels is not None:
            loss = nn.CrossEntropyLoss()(logits, labels)

        return SequenceClassifierOutput(loss=loss, logits=logits)

    def save_pretrained(self, path):
        os.makedirs(path, exist_ok=True)
        self.encoder.save_pretrained(os.path.join(path, 'encoder'))
        torch.save(self.head.state_dict(), os.path.join(path, 'classifier_head.pt'))
        self.config.save_pretrained(path)

print('Loading DNABERT-2 encoder...')
model = DNABertClassifier(MODEL_NAME).to(device)
total_params = sum(p.numel() for p in model.parameters())
train_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'✓ {total_params:,} total params  |  {train_params:,} trainable')

## 8. Training Setup

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = torch.softmax(torch.tensor(logits), dim=-1).numpy()
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy': accuracy_score(labels, preds),
        'f1':       f1_score(labels, preds, average='binary'),
        'auroc':    roc_auc_score(labels, probs[:, 1]),
    }

training_args = TrainingArguments(
    output_dir           = './dnabert2_checkpoints',
    num_train_epochs     = EPOCHS,
    per_device_train_batch_size = BATCH_SIZE,
    per_device_eval_batch_size  = 32,
    eval_strategy        = 'epoch',
    save_strategy        = 'epoch',
    logging_steps        = 50,
    learning_rate        = LR,
    weight_decay         = 0.01,
    warmup_ratio         = 0.1,
    fp16                 = True,
    load_best_model_at_end = True,
    metric_for_best_model  = 'auroc',
    greater_is_better    = True,
    report_to            = 'none',
    seed                 = SEED,
)

trainer = Trainer(
    model           = model,
    args            = training_args,
    train_dataset   = train_dataset,
    eval_dataset    = test_dataset,
    compute_metrics = compute_metrics,
)

print('✓ Trainer configured')
print(f'  Epochs: {EPOCHS}  |  Batch: {BATCH_SIZE}  |  LR: {LR}')
print(f'  Train steps/epoch: {len(train_dataset) // BATCH_SIZE}')

## 9. Training
~6-7 min on T4 for 3 epochs × ~24k training samples.

In [ ]:
print('🚀 Training started...\n')
result = trainer.train()

mins = result.metrics['train_runtime'] / 60
print(f'\n✓ Training complete in {mins:.1f} min')
print(f'  Final train loss: {result.metrics["train_loss"]:.4f}')

## 10. Evaluation

In [ ]:
print('📊 Evaluating on test set...\n')
metrics = trainer.evaluate()

print('=' * 45)
print('  EVALUATION RESULTS')
print('=' * 45)
print(f'  Accuracy : {metrics["eval_accuracy"]:.4f}')
print(f'  F1 Score : {metrics["eval_f1"]:.4f}')
print(f'  AUROC    : {metrics["eval_auroc"]:.4f}')
print('=' * 45)

# Detailed report
pred_out = trainer.predict(test_dataset)
preds  = np.argmax(pred_out.predictions, axis=-1)
labels = pred_out.label_ids

print('\n' + classification_report(
    labels, preds, target_names=['Benign', 'Pathogenic']
))
print('Confusion Matrix:')
print(confusion_matrix(labels, preds))

## 11. Save Model

In [ ]:
SAVE_DIR = './dnabert2_clinvar_final'
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print(f'✓ Model saved to {SAVE_DIR}')

# Zip for download
!zip -rq dnabert2_clinvar_model.zip $SAVE_DIR
print('✓ Zipped → dnabert2_clinvar_model.zip')

from google.colab import files
files.download('dnabert2_clinvar_model.zip')

## 12. Inference

In [ ]:
def predict(sequence: str) -> dict:
    """Predict pathogenicity of a DNA sequence.

    Args:
        sequence: DNA string (A/C/G/T), ideally 129 bp.

    Returns:
        Dict with Benign and Pathogenic probabilities.
    """
    model.eval()
    inputs = tokenizer(
        sequence,
        max_length=MAX_LENGTH,
        padding='max_length',
        truncation=True,
        return_tensors='pt',
    ).to(device)

    with torch.no_grad():
        logits = model(**inputs).logits
        probs  = torch.softmax(logits, dim=-1).squeeze().cpu().numpy()

    return {'Benign': float(probs[0]), 'Pathogenic': float(probs[1])}


# ── Demo: run inference on 5 test samples ──
print('🔬 Inference demo:\n')
for i in range(5):
    row = test_df.iloc[i]
    true = 'Pathogenic' if row['label'] == 1 else 'Benign'
    result = predict(row['sequence'])
    pred_label = max(result, key=result.get)
    marker = '✓' if pred_label == true else '✗'
    print(
        f'{marker}  Gene: {row["GeneSymbol"]:>10s}  '
        f'True: {true:>11s}  '
        f'Pred: {pred_label:>11s}  '
        f'(B={result["Benign"]:.3f}, P={result["Pathogenic"]:.3f})'
    )